# Dataset Audit - VPR
Das Notebook prüft die Daten auf Qualität und Konsistenz.  
Ergebnisse sollen durch diese Datenauswertung besser zu verstehen und zu analysieren sein.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.dpi"] = 300

# Beispiel einer Datarow pro Bild in metadata.parquet 
# {"image_id":387094212494432,
# "sequence_id":"205na8blq96arez7kzi15n",
# "captured_at":1567325430690,
# "lat":52.29240111837573,
# "lon":7.932048439979553,
# "compass_angle":285.85879516602,
# "is_pano":false,
# "creator_id":109122484655498,
# "split":"database"}

def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")

PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures" / "dataset"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH_META = PROCESSED_DIR / "metadata.parquet"
DATA_PATH_TRAIN = PROCESSED_DIR / "train_sequences.txt"
DATA_PATH_QUERY = PROCESSED_DIR / "query_sequences.txt"
DATA_PATH_BASE = PROCESSED_DIR / "database_sequences.txt"

metadata = pd.read_parquet(DATA_PATH_META)

print(f"Metadaten geladen: {len(metadata):,} Bilder")


pos = pd.read_csv(PROCESSED_DIR / "positive_candidates.csv")
print(f"Positives gesamt:        {len(pos):,}")
print(f"davon < 1 m:             {(pos['distance_m'] < 1).sum():,}")
print(
    f"Queries mit Positives:   {pos['query_image_id'].nunique():,} von {(metadata['split'] == 'query').sum():,}"
)
print(pos.groupby("query_image_id").size().describe())




# Grundstatistik

Vergleich von Mittelwert und Median:
- Median $<_{ deutlich}$ Mittelwert - sehr lange Strecken ziehen den Mittelwert künstlich hoch
- Median $>_{ deutlich}$ Mittelwert - sehr kurze Strecken ziehen den Mittelwert künstlich runter
  

Besondere Sequenzen:
- Sequenzen mit #=1 Bild **nicht** zwangshaft enfernen aber analysiert werden

In [ ]:

# Bilder pro Sequenz
images_per_sequence = metadata["sequence_id"].value_counts()

print("=" * 50)
print("DATASET STATISTIK")
print("=" * 50)

print(f"Bilder insgesamt:       {metadata['image_id'].nunique()}")
print(f"Sequenzen insgesamt:    {metadata['sequence_id'].nunique()}")

print("Bilder pro Sequenz")
print(f"Median                  {images_per_sequence.median():.0f}")
print(f"Minimum                 {images_per_sequence.min():.0f}")
print(f"Maximum                 {images_per_sequence.max():.0f}")
print(f"Mittelwert              {images_per_sequence.mean():.0f}")

print ("=" * 50)

# Fehlende Werte
Für jede Metaddatenspalte wird die Anzahl und der Prozentteil berechnet.
  
Fehlende Werte werden **nicht automatisch** als Löschungsgrund angesehen.



In [ ]:
print("Spalte       Missing         %")
print("-" * 40)

for column in metadata.columns:
    missing_count = metadata[column].isna().sum()
    missing_percent = missing_count / len(metadata) * 100
    print(f"{column:<15} {missing_count:<10} {missing_percent:.2f}%")



# Sequenzgrößen
10 längsten und 10 kürzesten werden ausgegeben und dann im Histogramm dargestellt.
  
Das Histogramm wird auf 1000 Bilder / Sequenz begrenzt damit die Verteilung anschaulich ist

In [ ]:

# to_string entfernt Name: count, dtype: int64 - rename_axis die sequence_id zeile
print("10 längste Sequenzen:")
print("-" * 40)
print(images_per_sequence.head(10).rename_axis(None).to_string())

print("\n")

print("10 kürzeste Sequenzen:")
print("-" * 40)
print(images_per_sequence.sort_values().rename_axis(None).head(10).to_string())



# Plotting
plt.figure(figsize=(10, 6))
plt.xlim(0, 1000)
plt.hist(images_per_sequence, bins=50)
plt.xlabel("Bilder pro Sequenz")
plt.ylabel("Anzahl Sequenzen")
plt.title("Verteilung der Sequenzgrößen")
plt.savefig(FIGURE_DIR / "sequence_sizes.png")
plt.show()

# Sequenzen Überschneidung 


Die 3 Sequenz Dateien werden eingelsen und verglichen. Es wird überprüft ob Sequenzen in Unterschiedlichen Dateien vorkommt.
  


In [ ]:
# Erstelle Sequenzen
train_sequences = pd.read_csv(DATA_PATH_TRAIN, header = None)[0]
query_sequences = pd.read_csv(DATA_PATH_QUERY, header = None)[0]
database_sequences = pd.read_csv(DATA_PATH_BASE, header = None)[0]


# In sets konvertieren
train_set = set(train_sequences)
query_set = set(query_sequences)
database_set = set(database_sequences)

train_database_overlap = train_set & database_set
train_query_overlap = train_set & query_set
query_database_overlap = query_set & database_set


# Überschneidungen Überprüfen
print("Überschneidung")
print("-" * 40)
print("Train geschnitten Database:  ", train_database_overlap)
print("Train geschnitten Query:     ", train_query_overlap)
print("Query geschnitten Databse:   ", query_database_overlap)
print("\n")


# Biler pro Split
images_per_split = metadata["split"].value_counts()
print("Bilder pro Split:")
print("_"*40)
print(images_per_split)
print("\n")


# Überprüfen ob diese mit unseren übereinstimmen
sequence_split = {}

for sequence in train_set:
    sequence_split[sequence] = "train"
for sequence in database_set:
    sequence_split[sequence] = "database"
for sequence in query_set:
    sequence_split[sequence] = "query"


# Konvert into Panda Split
expected_split = pd.Series(sequence_split, name = "expected_split")


# pro Sequence genau einen Split
metadata_sequence_split = (
    metadata[["sequence_id", "split"]].assign(sequence_id = lambda df : df["sequence_id"].astype(str)).drop_duplicates()
)


# Prüfen, ob eine Sequenz in den Metadaten mehr als einen Split besitzt
split_counts_per_sequence = metadata_sequence_split.groupby("sequence_id")["split"].nunique()
multiple_metadata_splits = split_counts_per_sequence[split_counts_per_sequence > 1]
metadata_sequence_split = metadata_sequence_split.set_index("sequence_id")


# Comparison Outer Join gibt alle Sequenzen zurück welche einer der Splits nicht enthält
comparison = metadata_sequence_split.join(expected_split, how = "outer")


# Missing Varibales
missing_in_split_files = comparison[comparison["expected_split"].isna()]
missing_in_metadata = comparison[comparison["split"].isna()]
comparable = comparison.dropna(subset = ["split", "expected_split"])
mismatched = comparable[comparable["split"] != comparable["expected_split"]]


print("-" * 40)
print(f"Sequenzen mit mehrern Splits:   {len(multiple_metadata_splits)}")
print(f"Abweichende Sequnezen:          {len(mismatched)}")
print(f"Nicht in Split-Dateien:         {len(missing_in_split_files)}")
print(f"Nicht in Metadata:              {len(missing_in_metadata)}")

# Zeitliche Analyse
captured_at liegt in Millisekunden und wird in einen Timestamp umgewandelt
  
Temporal Shift != Fehler -> zeigt wie gut das VPR auf Time Shift arbeiten kann

In [ ]:
captured = pd.to_datetime(
    metadata["captured_at"],
    unit = "ms",
    errors = "coerce"
)

metadata["captured_datetime"] = captured
metadata["year"] = metadata["captured_datetime"].dt.year


print("Exakte Bilderanzahl pro Jahr")
print("-" * 40)
print(metadata["year"].value_counts().rename_axis(None).sort_index().to_string())
print()

year_spilt = metadata.groupby(["year", "split"]).size()
print(year_spilt)


# Plotting
year_counts = metadata["year"].value_counts().sort_index()
plt.figure(figsize = (12,6))
plt.bar(year_counts.index, year_counts.values)
plt.xlabel("Jahr")
plt.ylabel("Anzahl Bilder")
plt.title("Mapillary Bilder je Aufnahmejahr")
plt.xticks(year_counts.index, rotation = 45)
plt.savefig(FIGURE_DIR / "images_per_year.png")
plt.show()

# Räumliche Analyse

Aus **lat** und **lon** werden Geodaten erzeugt



In [ ]:
# Geodata -> lat, lon
# Jedes Bild soll - Bild (lat lon) haben
images_gdf = gpd.GeoDataFrame(
    metadata,
    geometry = gpd.points_from_xy(
        metadata["lon"],
        metadata["lat"]
    ),
    crs = "EPSG:4326"
)

# Plotting Mapillary Coverage Osnabrück
ax = images_gdf.plot( figsize = (10, 10), markersize = 1)
ax.set_title("Mapillary Coverage Osnabrück")
plt.savefig(FIGURE_DIR / "coverage_map.png")
plt.show()


# Plotting Datbase vs Query Coverage
fig, ax = plt.subplots(figsize=(10, 10))
images_gdf[images_gdf["split"] == "database"].plot(ax=ax, markersize=1, label="Database")
images_gdf[images_gdf["split"] == "query"].plot(ax=ax, markersize=1, label="Query")
ax.set_title("Database vs Query")
ax.legend()
plt.savefig(FIGURE_DIR / "database_vs_query.png")
plt.show()

In [ ]:
pos = pd.read_csv(PROCESSED_DIR / "positive_candidates.csv")

# metadata nach image_id indizieren, damit .join darauf zugreifen kann
meta_idx = metadata.set_index("image_id")[["captured_at", "creator_id", "sequence_id"]]

pos = pos.join(meta_idx.add_prefix("q_"), on="query_image_id").join(
    meta_idx.add_prefix("db_"), on="database_image_id"
)

pos["same_creator"] = pos["q_creator_id"] == pos["db_creator_id"]
pos["same_sequence"] = pos["q_sequence_id"] == pos["db_sequence_id"]
pos["dt_minutes"] = (pos["q_captured_at"] - pos["db_captured_at"]).abs() / 60_000
pos["dt_days"] = pos["dt_minutes"] / (60 * 24)

n = len(pos)
print("=" * 58)
print("NEAR-DUPLICATE-ANALYSE DER POSITIVES")
print("=" * 58)
print(f"Positive-Paare gesamt:          {n:,}")
print(
    f"Queries mit Positives:          {pos['query_image_id'].nunique():,} "
    f"von {(metadata['split'] == 'query').sum():,}"
)
print(
    f"Positives pro Query (Median):   {pos.groupby('query_image_id').size().median():.0f}"
)
print()
print(
    f"Abstand < 1 m:                  {(pos['distance_m'] < 1).sum():>9,}  ({(pos['distance_m'] < 1).mean() * 100:5.1f} %)"
)
print(
    f"Gleicher creator_id:            {pos['same_creator'].sum():>9,}  ({pos['same_creator'].mean() * 100:5.1f} %)"
)
print(
    f"Gleiche sequence_id:            {pos['same_sequence'].sum():>9,}  ({pos['same_sequence'].mean() * 100:5.1f} %)"
)
print()
print(
    f"Aufnahme < 10 Minuten auseinander:  {(pos['dt_minutes'] < 10).sum():>9,}  ({(pos['dt_minutes'] < 10).mean() * 100:5.1f} %)"
)
print(
    f"Aufnahme < 1 Tag auseinander:       {(pos['dt_days'] < 1).sum():>9,}  ({(pos['dt_days'] < 1).mean() * 100:5.1f} %)"
)
print(
    f"Aufnahme > 180 Tage auseinander:    {(pos['dt_days'] > 180).sum():>9,}  ({(pos['dt_days'] > 180).mean() * 100:5.1f} %)"
)
print()
print("Zeitabstand in Tagen:")
print(pos["dt_days"].describe([0.25, 0.5, 0.75, 0.95]).to_string())
print("=" * 58)


# Audit Fazit
  
Ein Datensatz gilt als gültig wenn folgende Bed. erfüllt sind:
  
- keine Fehlende Werte die notwendig sind
- keine Überscheidungen der Sequenzen in Train, Query und Databse
- jede Sequenz ist in den Split Dateien vorhanden
- Bildzahlen addieren zur Gesamtzahl





In [ ]:
# Wenn unter 2% ignorieren
print("Panorama-Verteilung:")
print(metadata.groupby(["split", "is_pano"]).size().unstack(fill_value=0).to_string())
print()
print(f"Panorama-Anteil gesamt: {metadata['is_pano'].mean() * 100:.1f} %")


audit = {
    "images": len(metadata),
    "sequences": metadata["sequence_id"].nunique(),
    "single_image_sequences": int((images_per_sequence == 1).sum()),
    "missing_values_total": int(
        metadata.drop(columns=["captured_datetime", "year"], errors="ignore")
        .isna()
        .sum()
        .sum()
    ),
    "split_sequence_overlap": (
        len(train_database_overlap)
        + len(train_query_overlap)
        + len(query_database_overlap)
    ),
    "metadata_sequence_split_mismatches": len(mismatched),
    "sequences_missing_in_split_files": len(missing_in_split_files),
    "sequences_missing_in_metadata": len(missing_in_metadata),
    "sequences_with_multiple_metadata_splits": len(multiple_metadata_splits),
    "split_image_sum_matches_total": bool(images_per_split.sum() == len(metadata)),
}

print("=" * 50)
print("FINALER DATASET AUDIT")
print("=" * 50)
for key, value in audit.items():
    print(f"{key:<45} {value}")
print("=" * 50)